In [ ]:
import os
import sys

import tensorflow as tf
import yaml

sys.path.append(os.path.abspath("../"))
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import glob

from train.models import FullModel
from util import dataset as u_dataset
from util import image as u_image

In [ ]:
batch_size = 32

model_name = "20251030-154803.keras"
path_to_models = "../../models/finished/"

In [ ]:
# Get encoder config
def load_config(config_path):
    with open(config_path) as f:
        config = yaml.safe_load(f)
    return config


config = load_config(f"../../logs/fit/{model_name.split('.')[0]}/config.yaml")

encoder_architecture = config["model"]["encoder"]["architecture"]
classifier_architecture = config["model"]["classifier"]["architecture"]
categories_config = config["categories"]

print(encoder_architecture)
print(classifier_architecture)


#### Load Test Data


In [ ]:
path_to_test_data = glob.glob("../../data/tfrecords/test_ds*.tfrecords")

test_ds = u_dataset.get_dataset(path_to_test_data)
test_ds = test_ds.batch(batch_size, drop_remainder=False)

#### Load Model

In [ ]:
model = FullModel.load(
    encoder_architecture,
    classifier_architecture,
    input_dims=config["model"]["encoder"]["input_dims"],
    filepath=path_to_models,
    filename=model_name,
    n_context=config["model"]["encoder"]["n_context"],
    only_train_encoder=config["model"]["encoder"]["only_train_encoder"],
    classifier_offsets=config["model"]["classifier"]["with_offsets"],
    encoder_only=False,
    verbose=True,
    n_meta=config["model"]["classifier"]["n_meta"],
    encoder_use_batch_norm=config["model"]["encoder"]["use_batch_norm"],
    classifier_use_batch_norm=config["model"]["classifier"]["use_batch_norm"],
    categories_config=config["categories"],
)
model.compile(optimizer=tf.keras.optimizers.Adam(), jit_compile=False)


#### Predict Data

In [ ]:
groundtruth_dataset = []
for x in test_ds:
    groundtruth_dataset.append(x)

# Take only image, camera, intrinsics for input
input_data = test_ds.map(lambda x: (x["image"], x["camera"], x["intrinsics"]))

# Convert tf.Data.Dataset into list
input_list = []
for x, y, z in input_data:
    input_list.append((x, y, z))

model.run_eagerly = True

predictions = [model.predict(x=batch) for batch in input_list]
metrics_list = [model.evaluate(x=batch, return_dict=True) for batch in groundtruth_dataset]

#### Evaluate Models

In [ ]:
# Calculate mean for metrics over each batch
keys = metrics_list[0].keys()
stacked_values = {key: tf.stack([metrics[key] for metrics in metrics_list]) for key in keys}
metrics_mean = {key: tf.reduce_mean(stacked_values[key]) for key in keys}

# Print Metrics
for key, value in metrics_mean.items():
  print(f"{key}: {value.numpy():.4f}")

### Accuracy, Precision, Recall

In [ ]:
# keys = predictions[0]["results"]["ball"].keys()
# print(keys)
# stacked_ball_predictions = {key: tf.stack([prediction["results"]["ball"][key] for prediction in predictions]) for key in keys}


# print(len(predictions))